In [ ]:
# Cell 1
import sys

!{sys.executable} -m pip install openai pandas openpyxl tqdm

In [ ]:
# Cell 2 Imports
import os, re, json, glob, hashlib
import pandas as pd
from tqdm import tqdm
from openai import OpenAI


In [ ]:
# Cell 3 Config
MODEL_NAME='openai/gpt-5.2-chat'
NUM_VOTES=3
REQ_FILE='requirements.xlsx'
RESULTS_DIR='generated_diagrams'
CACHE_DIR='cache'
os.makedirs(CACHE_DIR,exist_ok=True)


In [ ]:
# Cell 4 OpenRouter
client=OpenAI(
 api_key=os.getenv('OPENROUTER_API_KEY'),
 base_url='https://openrouter.ai/api/v1'
)


In [ ]:
# Cell 5 Load Requirements
df=pd.read_excel(REQ_FILE)
df.head()


In [ ]:
# Cell 6 Cache Helpers
def load_cache(path):
    if os.path.exists(path):
        with open(path,'r',encoding='utf8') as f:
            return json.load(f)
    return None

def save_cache(path,obj):
    os.makedirs(os.path.dirname(path),exist_ok=True)
    with open(path,'w',encoding='utf8') as f:
        json.dump(obj,f,indent=2)


In [ ]:
import time

def call_llm(prompt):

    for attempt in range(3):

        try:

            response = (
                client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role":"user",
                            "content":prompt
                        }
                    ],
                    timeout=120
                )
            )

            return (
                response
                .choices[0]
                .message
                .content
            )

        except Exception as e:

            print(
                f"Retry {attempt+1}: {e}"
            )

            time.sleep(5)

    return "ERROR"

In [ ]:
# Cell 8 Requirement Atom Extraction
REQ_PROMPT='''Extract atomic requirements as JSON from:\n{context}'''
def get_requirement_atoms(req_id,functional,usecase):
    path=f'cache/requirement_atoms/{req_id}.json'
    c=load_cache(path)
    if c:return c
    context=f'Functional Requirements:\n{functional}\n\nUse Case:\n{usecase}'
    out=call_llm(REQ_PROMPT.format(context=context))
    try: data=json.loads(out)
    except: data={'raw':out}
    save_cache(path,data)
    return data


In [ ]:
# Cell 9 Diagram Discovery
def get_diagrams(req_id):
    return glob.glob(os.path.join(RESULTS_DIR,str(req_id),'*.puml'))


In [ ]:
# Cell 10 Read Diagram
def load_diagram(path):
    with open(path,'r',encoding='utf8') as f:
        return f.read()


In [ ]:
# Cell 11 Activity Atom Parser
def extract_activity_atoms(req_id,diagram_name,puml):
    path=f'cache/activity_atoms/{req_id}_{diagram_name}.json'
    c=load_cache(path)
    if c:return c
    actions=re.findall(r':(.*?);',puml)
    decisions=re.findall(r'if\s*\((.*?)\)',puml,re.I)
    swimlanes=list(set(re.findall(r'\|(.*?)\|',puml)))
    flows=[(actions[i],actions[i+1]) for i in range(len(actions)-1)]
    data={
      'actions':actions,
      'decisions':decisions,
      'swimlanes':swimlanes,
      'flows':flows
    }
    save_cache(path,data)
    return data


In [ ]:
# Cell 12 Holistic Check
HOLISTIC='''Compare activity diagram and requirements. Return JSON issues.'''
def holistic_check(req_id,diagram_name,requirements,diagram):
    path=f'cache/holistic/{req_id}_{diagram_name}.json'
    c=load_cache(path)
    if c:return c
    out=call_llm(HOLISTIC+'\n'+requirements+'\n'+diagram)
    save_cache(path,{'raw':out})
    return {'raw':out}


In [ ]:
def verify_actions(
    req_id,
    diagram_name,
    actions,
    requirements
):

    path = (
        f"cache/action_checks/"
        f"{req_id}_{diagram_name}.json"
    )

    cached = load_cache(path)

    if cached:
        return cached

    prompt = f"""
Requirements:

{requirements}

Actions:

{json.dumps(actions, indent=2)}

For each action determine:

1. Supported by requirements?
2. Missing details?
3. Hallucinated?
4. Actor mismatch?

Return JSON.
"""

    result = call_llm(prompt)

    save_cache(
        path,
        {"raw": result}
    )

    return {"raw": result}

In [ ]:
def verify_decisions(
    req_id,
    diagram_name,
    decisions,
    requirements
):

    path = (
        f"cache/decision_checks/"
        f"{req_id}_{diagram_name}.json"
    )

    cached = load_cache(path)

    if cached:
        return cached

    prompt = f"""
Requirements:

{requirements}

Decision Nodes:

{json.dumps(decisions, indent=2)}

Check:

1. Correct conditions
2. Missing branches
3. Wrong branches
4. Unsupported decisions

Return JSON.
"""

    result = call_llm(prompt)

    save_cache(
        path,
        {"raw": result}
    )

    return {"raw": result}

In [ ]:
def verify_flows(
    req_id,
    diagram_name,
    flows,
    requirements
):

    path = (
        f"cache/flow_checks/"
        f"{req_id}_{diagram_name}.json"
    )

    cached = load_cache(path)

    if cached:
        return cached

    prompt = f"""
Requirements:

{requirements}

Flows:

{json.dumps(flows, indent=2)}

Check:

1. Ordering correctness
2. Missing activities
3. Invalid transitions

Return JSON.
"""

    result = call_llm(prompt)

    save_cache(
        path,
        {"raw": result}
    )

    return {"raw": result}

In [ ]:
def verify_swimlanes(
    req_id,
    diagram_name,
    swimlanes,
    requirements
):

    path = (
        f"cache/swimlane_checks/"
        f"{req_id}_{diagram_name}.json"
    )

    cached = load_cache(path)

    if cached:
        return cached

    prompt = f"""
Requirements:

{requirements}

Swimlanes:

{json.dumps(swimlanes, indent=2)}

Check:

1. Correct actor allocation
2. Missing actors
3. Hallucinated actors

Return JSON.
"""

    result = call_llm(prompt)

    save_cache(
        path,
        {"raw": result}
    )

    return {"raw": result}

In [ ]:
def verify_requirement_atoms(
    req_id,
    diagram_name,
    req_atoms,
    diagram
):

    path = (
        f"cache/req_atom_checks/"
        f"{req_id}_{diagram_name}.json"
    )

    cached = load_cache(path)

    if cached:
        return cached

    prompt = f"""
Requirement Atoms:

{json.dumps(req_atoms, indent=2)}

Diagram:

{diagram}

Check whether each requirement
atom is:

1. Implemented
2. Partially implemented
3. Missing

Return JSON.
"""

    result = call_llm(prompt)

    save_cache(
        path,
        {"raw": result}
    )

    return {"raw": result}

In [ ]:
def diagram_checks(
    req_id,
    diagram_name,
    atoms,
    requirements
):

    return {

        "actions":
            verify_actions(
                req_id,
                diagram_name,
                atoms["actions"],
                requirements
            ),

        "decisions":
            verify_decisions(
                req_id,
                diagram_name,
                atoms["decisions"],
                requirements
            ),

        "flows":
            verify_flows(
                req_id,
                diagram_name,
                atoms["flows"],
                requirements
            ),

        "swimlanes":
            verify_swimlanes(
                req_id,
                diagram_name,
                atoms["swimlanes"],
                requirements
            )
    }

In [ ]:
def requirement_checks(
    req_id,
    diagram_name,
    req_atoms,
    diagram
):

    return verify_requirement_atoms(
        req_id,
        diagram_name,
        req_atoms,
        diagram
    )

In [ ]:
# Cell 20 MCeT-X
def cross_check(req_id,diagram_name,holistic,diagram_res,req_res):
    path=f'cache/crosscheck/{req_id}_{diagram_name}.json'
    c=load_cache(path)
    if c:return c
    out=call_llm('Cross-check issues and remove contradictions')
    save_cache(path,{'raw':out})
    return {'raw':out}


In [ ]:
# Cell 21 Full Verification
def verify_diagram(req_id,functional,usecase,diagram_path):
    diagram_name=os.path.basename(diagram_path)
    print(
        f"Processing "
        f"{req_id} "
        f"{diagram_name}"
    )
    final_path=f'cache/final/{req_id}_{diagram_name}.json'
    c=load_cache(final_path)
    if c:return c
        
    print("Requirement atoms...")
    req_atoms=get_requirement_atoms(req_id,functional,usecase)
    diagram=load_diagram(diagram_path)
    print("Activity atoms...")
    atoms=extract_activity_atoms(req_id,diagram_name,diagram)

    req_text=f'{functional}\n\n{usecase}'
    print("Holistic...")
    hol=holistic_check(req_id,diagram_name,req_text,diagram)
    print("Diagram checks...")
    dia=diagram_checks(req_id,diagram_name,atoms,req_text)
    print("Requirement checks...")
    req=requirement_checks(req_id,diagram_name,req_atoms,diagram)
    print("Cross-check...")
    cross=cross_check(req_id,diagram_name,hol,dia,req)

    result={
      'RequirementID':req_id,
      'DiagramName':diagram_name,
      'ActionIssues':dia['actions'],
      'DecisionIssues':dia['decisions'],
      'FlowIssues':dia['flows'],
      'SwimlaneIssues':dia['swimlanes'],
      'RequirementAtomIssues':req,
      'HolisticIssues':hol,
      'CrossChecked':cross
    }
    save_cache(final_path,result)
    return result


In [ ]:
# Cell 22 Run All
results=[]
for _,row in tqdm(df.iterrows(),total=len(df)):
    req_id=row['RequirementID']
    functional=str(row['FunctionalRequirements'])
    usecase=str(row['UseCaseScenarios'])

    diagrams=get_diagrams(req_id)

    for d in diagrams:
        try:
            results.append(
                verify_diagram(req_id,functional,usecase,d)
            )
        except Exception as e:
            print(req_id,d,e)


In [ ]:
# Cell 23 Export
flat=[]
for r in results:
    flat.append({
      'RequirementID':r['RequirementID'],
      'DiagramName':r['DiagramName'],
      'ActionIssues':json.dumps(r['ActionIssues']),
      'DecisionIssues':json.dumps(r['DecisionIssues']),
      'FlowIssues':json.dumps(r['FlowIssues']),
      'SwimlaneIssues':json.dumps(r['SwimlaneIssues'])
    })
pd.DataFrame(flat).to_excel('mcet_ad_results.xlsx',index=False)
print('Saved')
